In this notebook we'll consider which factors affect our choice of embedding model.

### Embedding model performance

The Massive Text Embedding Benchmark (MTEB) is a standardised leaderboard for comparing embedding models across retrieval, clustering and classification tasks. According to the LangChain docs (https://docs.langchain.com/oss/python/integrations/embeddings), this it the de-facto industry reference.

However, it could be worth considering smaller models that are trained on domain-specific data.

### Local vs externally hosted

If instead we were using a hosted model we'd want to consider some other factors. 

#### Data Privacy & Security

Sending documents containing sensitive information to an external embedding API might not be an option for some companies. However, a platform like Google's Vertex AI would enable a Google embedding model to be deployed on the company's own internal infrastructure.

When using a hosted model, it's important to consider what would happen if the third-party makes changes to the model, e.g updates, deprecation or increased price.

#### Cost & Latency

The top-performing models (according to MTEB) cost between ~$0.01 - $0.1 per million tokens. The providers of such models include Google, OpenAI and Cohere, with Google's `gemini-embedding-001` at the top of the list.

In addition to the indexing phase, in which each document chunk is embedded and added to the vector store, the embedding model also gets run on each user query. During indexing latency might not be a concern, but if the embedding model is slow to embed queries, this could slow down inference.

### This Project

For the purpose of this project we want to run the embedding model locally, on CPU, which limits our choices. Looking at MTEB, the smallest model with the best performance on retrieval tasks is `BAAI/bge-small-en-v1.5`, so we'll go with that for now. 

When setting the chunk size, we opted for 1000 in order to comply with a model with a maximum token length of 256. The model we've chosen has a maximum token length of 512, so in theory we could double the chunk size, but we could revisit this in the future.


### Embedding model: BAAI/bge-small-en-v1.5

BGE models are trained for asymmetric retrieval: queries and documents are encoded differently. 

Queries need to be prefixed with the following instruction for best retrieval quality: *Represent this sentence for searching relevant passages:*.

In [ ]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from rag_pipeline.config import SampleDocsConfig, ChunkingConfig, EmbeddingModelConfig

from rag_pipeline.parse_documents import parse_document
from rag_pipeline.chunking import chunk_documents

In [ ]:
sample_docs_config = SampleDocsConfig()
chunking_config = ChunkingConfig()

parsed_docs = parse_document(sample_docs_config.doc1_path)
doc_chunks = chunk_documents(
    parsed_docs, chunking_config.chunk_size, chunking_config.chunk_overlap
)

In [ ]:
config = EmbeddingModelConfig()


def create_embedding_model() -> HuggingFaceEmbeddings:
    """Return a configured HuggingFaceEmbeddings instance for BGE-small.

    normalize_embeddings=True makes cosine similarity equivalent to dot product
    on the resulting vectors — slightly faster at search time and the recommended
    setting for BGE models.

    The model is downloaded from HuggingFace on first use and cached locally;
    subsequent calls load from cache.
    """
    return HuggingFaceEmbeddings(
        model_name=config.model_name,
        encode_kwargs={"normalize_embeddings": config.normalize_embeddings},
    )

In [ ]:
def embed_chunks(
    chunks: list[Document],
    model: HuggingFaceEmbeddings,
) -> list[list[float]]:
    """Return one embedding vector per chunk, in input order."""
    return model.embed_documents([chunk.page_content for chunk in chunks])

In [ ]:
embedding_model = create_embedding_model()
chunk_embeddings = embed_chunks(doc_chunks, embedding_model)

In [ ]:
expected_embedding_dim = 384
expected_num_chunks = 15

assert type(chunk_embeddings) == list
assert len(chunk_embeddings) == expected_num_chunks
assert len(chunk_embeddings[0]) == expected_embedding_dim